# LabCorp Coverage Check for NBA 3.1 Lab Input Query

Checks whether the Lab query in Cell 72 of `01_Input_data_creation` (NBA 3.1 - Dev) picks up LabCorp/Prognos rows (`LAB_IDENTIFIER = 'P'`) as well as Quest rows (`'Q'`) from `PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST`, and how many LabCorp rows each filter removes. Numbers below were run on 2026-11-24 against the load dated 2026-10-20.

## Findings

**Answer: yes, Cell 72 uses LabCorp data.** It has no `LAB_IDENTIFIER` filter, so Quest (`Q`) and LabCorp/Prognos (`P`) rows both go in. After all the SQL filters, LabCorp makes up **52% of the output rows** (604.9M of 1,163.2M). The query does not tag rows by source, though, and many LabCorp rows are dropped silently because their patient is not linked.

### 1. Source profile (whole table)

`LAB_IDENTIFIER` has only two values, `P` (1,838,031,746 rows) and `Q` (896,489,914 rows). Both are exactly one character with no whitespace. There are 0 NULL, 0 blank and 0 other values, and P + Q equals the 2,734,521,660 total rows (see cell 0).

| Metric | LabCorp / Prognos (P) | Quest (Q) |
| --- | --- | --- |
| Rows | 1,838.0M | 896.5M |
| Distinct PATIENT_ID | 40.0M | 76.2M |
| Latest SVC_DT | 2026-10-05 | 2026-09-30 |
| Latest DATA_DT / CRTD_DT | 2026-10-20 | 2026-10-20 |
| Rows with a DV_TOKEN_1 | 100% | 0% |
| Rows with a usable LAAD PATIENT_ID (numeric, not -1) | 43.0% | 94.6% |

This matches the stated context: LabCorp runs to Oct 2026, Quest to Sep 2026, last load 2026-10-20, and the Prognos tokens are only on P rows. There are about 40 P rows with garbage dates (`0001-01-01`, `2001`) or dates that can't be parsed. They are dropped by the `SVC_DT <= lookback_end` / null-date logic or don't matter.

### 2. PATIENT_ID linkage: worse than "some 2023 rows"

| SVC year | P rows | P linked (numeric ID) | % linked | P `PATIENT_ID IS NULL` | P `PATIENT_ID = '-1'` | Q % linked |
| --- | --- | --- | --- | --- | --- | --- |
| 2021 | 108.6M | 47.7M | 43.9% | 60.5M | 0.4M | 95.2% |
| 2022 | 340.6M | 144.8M | 42.5% | 194.9M | 0.9M | 95.0% |
| 2023 | 408.8M | 169.7M | 41.5% | 238.1M | 1.0M | 95.0% |
| 2024 | 639.7M | 259.9M | 40.6% | 378.2M | 1.6M | 95.0% |
| 2025 | 199.7M | 77.0M | 38.6% | 65.1M | 57.6M | 91.5% |
| 2026 | 140.6M | 90.4M | 64.3% | 0 | 50.2M | 92.4% |

* About 55–59% of LabCorp rows from **2021–2024** have a NULL `PATIENT_ID`, not only 2023.
* All 2026 LabCorp rows have a non-null `PATIENT_ID` and a token. But **35.7% of them (50.2M) have the placeholder `'-1'`**, which means there's no real LAAD link. The same happens to 28.8% of 2025 rows.
* `REGEXP_LIKE(PATIENT_ID, '^[0-9]+$')` in Cell 72 removes both NULL and `-1` (it doesn't match the minus sign). So unlinked rows are excluded correctly, but **57% of all LabCorp rows never enter the pipeline**.

### 3. Cell 72 filter waterfall by source (whole table, rows)

| Step | P rows | Q rows |
| --- | --- | --- |
| S0 All rows (after provider LEFT JOIN) | 1,838.3M | 896.8M |
| S1 Numeric PATIENT_ID | 789.8M (43.0%) | 848.0M (94.6%) |
| S2 Valid numeric test value | 786.4M | 811.9M |
| S3 Test type GFR/UACR or LOINC K/HbA1c | 786.4M | 811.8M |
| S4 UOM filter (GFR/UACR branch) | 786.4M | 797.4M |
| S5 Provider found in DIM_PRVDR | 772.1M | 752.0M |
| **S6 Final (PCP specialty list)** | **604.9M** | **558.3M** |
| of which GFR | 226.3M | 500.5M |
| of which UACR | 40.2M | 57.8M |
| of which Potassium 2823-3 / HbA1c 4548-4 | 338.5M | **0** |

* Almost all LabCorp loss happens at **S1 (patient linkage)**. The test-name, UOM and value filters remove close to nothing from P, so the P feed already contains only CKD-relevant tests.
* **Potassium and HbA1c come only from LabCorp.** Quest has no rows with those LOINCs, so the new 3.1 K/HbA1c events depend fully on the P feed.
* LabCorp makes up 31% of GFR rows and 41% of UACR rows.
* P rows from 2022–2023 have about 35M final rows with a NULL `LOINC_CD`. Cell 72 handles them with the TEST_TYPE → eGFR/UACR LOINC fallback in PySpark.

### 4. Final output by year (before HCP-universe and lookback_end filters)

| Year | P rows | P patients | Q rows | Q patients |
| --- | --- | --- | --- | --- |
| 2021 | 36.2M | 5.9M | 88.0M | 13.5M |
| 2022 | 110.1M | 11.7M | 154.4M | 22.3M |
| 2023 | 129.7M | 12.8M | 111.5M | 22.9M |
| 2024 | 199.8M | 12.9M | 121.8M | 24.5M |
| 2025 | 59.1M | 11.4M | 47.9M | 25.6M |
| 2026 | 70.0M | 18.3M | 34.7M | 21.9M |

Distinct patients across all years: P = 34.2M, Q = 60.8M, union = 88.2M. So **27.5M patients (31%) show up only through LabCorp**, and 6.7M appear in both sources.

### 5. Caveats and recommendations

* **No source tag:** Cell 72 does not select `LAB_IDENTIFIER`. Add it, or at least log the P/Q split under `RUN_DIAGNOSTICS`, so LabCorp coverage can be tracked over time.
* **Unlinked history:** pre-2025 LabCorp history is only about 41% linked. First-occurrence and recency features for those patients will start later than their true first test. If the Prognos `DV_TOKEN_*` columns can be bridged to LAAD IDs, that would recover the most coverage.
* **'-1' placeholder:** the regex already excludes it, but any other place that only checks `PATIENT_ID IS NOT NULL` would wrongly treat about 108M P rows (2025–2026) as one patient, `-1`.
* **Specialty filter:** the PCP-only list (`FM, GP, IM, NRP, PHA, …`) removes about 22% of P and 26% of Q rows that survive the earlier steps. Nephrology is not in the list. This applies to both sources and is not specific to LabCorp, but check that it is intended.
* Counts here are row-level from Snowflake, before the `MEDIAN` / `GROUP BY`, the `in_universe()` HCP filter, and `apply_event_list()`. Steps S0–S6 use the raw `DIM_PRVDR` join, which adds about 0.28M duplicate rows.

## Reproducible queries
Run the cells below to refresh the numbers. They use `Get_Data_Snowflakes` from the shared connection notebook. The whole-table waterfall takes about 5 minutes on `PROD_CYRUS_BI_WH`.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
LAB_TBL = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST"
PRV_TBL = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR"
DT   = "CASE WHEN l.SVC_DT LIKE '%/%/%' THEN TRY_TO_DATE(l.SVC_DT,'MM/DD/YYYY') ELSE TRY_TO_DATE(LEFT(l.SVC_DT,9),'DDMONYYYY') END"
SPEC = "('FM','FPG','GP','GPM','IFP','IM','IMG','IPM','NRP','PHA')"
UOM  = "('ML/MIN/1.73M2','MCG/MGCREAT','MG/G{CREAT}','MG/GCREAT','MG/GCR')"

In [0]:
# Every distinct LAB_IDENTIFIER value, with brackets/length to expose whitespace or hidden characters
lab_id_values = Get_Data_Snowflakes(f"""
SELECT LAB_IDENTIFIER,
       '[' || COALESCE(LAB_IDENTIFIER, '<NULL>') || ']' AS BRACKETED,
       LENGTH(LAB_IDENTIFIER) AS LEN,
       COUNT(*) AS N_ROWS,
       MIN(DATA_DT) AS MIN_DATA_DT, MAX(DATA_DT) AS MAX_DATA_DT
FROM {LAB_TBL}
GROUP BY 1, 2, 3
ORDER BY N_ROWS DESC""")
display(lab_id_values)

# Reconciliation: total rows must equal P + Q + NULL/blank + other
lab_id_recon = Get_Data_Snowflakes(f"""
SELECT COUNT(*) AS TOTAL_ROWS,
       SUM(IFF(LAB_IDENTIFIER = 'P', 1, 0)) AS P_ROWS,
       SUM(IFF(LAB_IDENTIFIER = 'Q', 1, 0)) AS Q_ROWS,
       SUM(IFF(LAB_IDENTIFIER IS NULL, 1, 0)) AS NULL_ROWS,
       SUM(IFF(TRIM(LAB_IDENTIFIER) = '', 1, 0)) AS BLANK_ROWS,
       SUM(IFF(LAB_IDENTIFIER NOT IN ('P', 'Q') AND TRIM(LAB_IDENTIFIER) <> '', 1, 0)) AS OTHER_ROWS
FROM {LAB_TBL}""")
display(lab_id_recon)

In [0]:
# Profile Datavant (DV_) token columns: fill rate, approx distinct, length, character pattern, by LAB_IDENTIFIER
TOKENS = ["DV_TOKEN_1", "DV_TOKEN_2", "DV_TOKEN_3", "DV_TOKEN_4", "DV_TOKEN_5", "DV_TOKEN_16", "DV_TOKEN_17"]
union_sql = "\nUNION ALL\n".join([f"""
SELECT LAB_IDENTIFIER, '{t}' AS TOKEN_COL,
       COUNT(*) AS N_ROWS,
       COUNT({t}) AS N_NON_NULL,
       ROUND(COUNT({t}) * 100.0 / COUNT(*), 2) AS PCT_FILLED,
       APPROX_COUNT_DISTINCT({t}) AS APPROX_DISTINCT,
       MIN(LENGTH({t})) AS MIN_LEN, MAX(LENGTH({t})) AS MAX_LEN,
       SUM(IFF(REGEXP_LIKE({t}, '^[A-Za-z0-9+/]+=*$'), 1, 0)) AS N_BASE64_LIKE,
       SUM(IFF(UPPER({t}) LIKE 'XXX%' OR {t} ILIKE '%INVALID%' OR {t} ILIKE '%NULL%', 1, 0)) AS N_PLACEHOLDER,
       LEFT(ANY_VALUE({t}), 8) || '...' AS EXAMPLE_PREFIX
FROM {LAB_TBL}
WHERE DATA_DT >= '2026-10-01'
GROUP BY 1""" for t in TOKENS])
token_profile = Get_Data_Snowflakes(union_sql + "\nORDER BY 1, 2")
display(token_profile)

# Do multiple patients share a token, and does one patient carry multiple tokens? (P rows, latest load)
token_vs_pid = Get_Data_Snowflakes(f"""
SELECT APPROX_COUNT_DISTINCT(PATIENT_ID) AS DISTINCT_PID,
       APPROX_COUNT_DISTINCT(DV_TOKEN_1) AS DISTINCT_TOKEN1,
       APPROX_COUNT_DISTINCT(PATIENT_ID || '|' || DV_TOKEN_1) AS DISTINCT_PID_TOKEN1_PAIRS,
       APPROX_COUNT_DISTINCT(IFF(PATIENT_ID = '-1', DV_TOKEN_1, NULL)) AS DISTINCT_TOKEN1_ON_UNLINKED
FROM {LAB_TBL}
WHERE LAB_IDENTIFIER = 'P' AND DATA_DT >= '2026-10-01'""")
display(token_vs_pid)

In [0]:
# Real tokens are 44-char base64 (SHA-256 sized); shorter values are Datavant 'XXX - ...' error tokens (missing/invalid PII)
TOKENS_FILLED = ["DV_TOKEN_1", "DV_TOKEN_2", "DV_TOKEN_3", "DV_TOKEN_4", "DV_TOKEN_17"]
err_sql = "\nUNION ALL\n".join([f"""
SELECT '{t}' AS TOKEN_COL, {t} AS ERROR_VALUE, COUNT(*) AS N_ROWS
FROM {LAB_TBL}
WHERE LAB_IDENTIFIER = 'P' AND DATA_DT >= '2026-10-01' AND LENGTH({t}) < 44
GROUP BY 1, 2""" for t in TOKENS_FILLED])
token_errors = Get_Data_Snowflakes(err_sql + "\nORDER BY 1, 3 DESC")
display(token_errors)

### DV_TOKEN_* columns: what they contain (latest load, `DATA_DT >= 2026-10-01`; 15.0M P rows, 7.2M Q rows)

The `DV_` columns hold **Datavant privacy-preserving linkage tokens**. Each one is a salted one-way hash of a specific combination of patient PII, encrypted with a key specific to the data supplier. They are not readable PII. Their purpose is to link the same patient across data sources without exposing identity.

| Column | Publicly cited Datavant definition | P fill | P approx distinct | Error tokens (P) | Q |
| --- | --- | --- | --- | --- | --- |
| DV_TOKEN_1 | Last name + first initial + gender + DOB | 100% | 5.02M | 0 | empty |
| DV_TOKEN_2 | Soundex(last) + Soundex(first) + gender + DOB | 100% | 4.87M | 0 | empty |
| DV_TOKEN_3 | Last name + first name + DOB + ZIP3 | 100% | 5.13M | 477 (`XXX - Z00000`) | empty |
| DV_TOKEN_4 | Last name + first name + gender + DOB | 100% | 5.08M | 0 | empty |
| DV_TOKEN_5 | SSN + gender + DOB | 0% | 0 | n/a | empty |
| DV_TOKEN_16 | SSN + first name | 0% | 0 | n/a | empty |
| DV_TOKEN_17 | No public definition (vendor-specific) | 100% | 4.96M | 400,793 (2.7%, mostly `XXX - A00000`) | empty |

* Valid tokens are always **44-character base64 strings**, which is the size of a SHA-256 hash.
* **The SSN-based tokens (5, 16) are empty.** Linkage depends only on name, gender, DOB and ZIP3 combinations.
* `XXX - ...` values are Datavant **error tokens**. They are written when a required PII input was missing or invalid, so treat them as NULL and never join on them. For example, 400K rows share `XXX - A00000` on Token 17.
* **Quest rows have no tokens at all.** Quest is linked only through `PATIENT_ID`.
* **Tokens vs LAAD ID (P):** 4.79M distinct `PATIENT_ID`s vs 5.02M distinct Token 1 values, with 5.00M distinct (ID, token) pairs, so the mapping is almost 1:1. Rows with `PATIENT_ID = '-1'` still carry **about 244K distinct Token 1 values**. These are patients that could be recovered if a token-to-LAAD bridge (e.g. a LAAD Datavant token table) is available.

Source for token definitions: [Establishing a framework for privacy-preserving record linkage among electronic health record and administrative claims databases within PCORnet®](https://link.springer.com/article/10.1186/s13104-022-06243-5) and [Real-World Matching Performance of Deidentified Record-Linking Tokens](https://pmc.ncbi.nlm.nih.gov/articles/PMC9474266/). Token 17's composition should be confirmed with IQVIA/Prognos.

In [0]:
linkage = Get_Data_Snowflakes(f"""
SELECT l.LAB_IDENTIFIER, YEAR({DT}) AS YR, COUNT(*) AS N_ROWS,
       SUM(IFF(REGEXP_LIKE(l.PATIENT_ID,'^[0-9]+$'),1,0)) AS LINKED_PID,
       SUM(IFF(l.PATIENT_ID = '-1',1,0))                AS PID_MINUS1,
       SUM(IFF(l.PATIENT_ID IS NULL,1,0))               AS PID_NULL,
       SUM(IFF(l.DV_TOKEN_1 IS NOT NULL,1,0))           AS HAS_TOKEN,
       MAX({DT}) AS MAX_SVC_DT, MAX(l.DATA_DT) AS MAX_DATA_DT
FROM {LAB_TBL} l
WHERE {DT} >= '2021-01-01'
GROUP BY 1, 2 ORDER BY 1, 2""")
display(linkage)

In [0]:
waterfall = Get_Data_Snowflakes(f"""
WITH base AS (
  SELECT l.LAB_IDENTIFIER,
    REGEXP_LIKE(l.PATIENT_ID,'^[0-9]+$') AS F_PID,
    (l.LCL_LAB_TEST_VAL_TXT IS NOT NULL AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%'
       AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%' AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%'
       AND REGEXP_LIKE(TRIM(l.LCL_LAB_TEST_VAL_TXT),'^[0-9]+(\\\\.[0-9]+)?$')) AS F_VAL,
    CASE WHEN UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%GFR%' OR UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%GLOMERULAR%' THEN 'GFR'
         WHEN UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%ALB%' OR UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%RATIO%' THEN 'UACR'
         ELSE 'OTHER' END AS TT,
    UPPER(REPLACE(l.LCL_LAB_TEST_RSLT_UOM_CD_CLEAN,' ','')) IN {UOM} AS F_UOM,
    l.LOINC_CD IN ('2823-3','4548-4') AS F_LOINC,
    d.NPI_NUMBER IS NOT NULL AS F_PRV,
    d.PRIMARY_SPECIALTY_CODE IN {SPEC} AS F_SPEC
  FROM {LAB_TBL} l
  LEFT JOIN {PRV_TBL} d ON l.NPI_ID = d.NPI_NUMBER
)
SELECT LAB_IDENTIFIER,
  COUNT(*) AS S0_ALL,
  SUM(IFF(F_PID,1,0)) AS S1_PID,
  SUM(IFF(F_PID AND F_VAL,1,0)) AS S2_VAL,
  SUM(IFF(F_PID AND F_VAL AND (TT<>'OTHER' OR F_LOINC),1,0)) AS S3_TEST,
  SUM(IFF(F_PID AND F_VAL AND ((TT<>'OTHER' AND F_UOM) OR F_LOINC),1,0)) AS S4_UOM,
  SUM(IFF(F_PID AND F_VAL AND ((TT<>'OTHER' AND F_UOM) OR F_LOINC) AND F_PRV,1,0)) AS S5_PRV,
  SUM(IFF(F_PID AND F_VAL AND ((TT<>'OTHER' AND F_UOM) OR F_LOINC) AND F_SPEC,1,0)) AS S6_FINAL,
  SUM(IFF(F_PID AND F_VAL AND TT='GFR'  AND F_UOM AND F_SPEC,1,0)) AS FINAL_GFR,
  SUM(IFF(F_PID AND F_VAL AND TT='UACR' AND F_UOM AND F_SPEC,1,0)) AS FINAL_UACR,
  SUM(IFF(F_PID AND F_VAL AND F_LOINC AND F_SPEC,1,0)) AS FINAL_K_HBA1C
FROM base GROUP BY 1 ORDER BY 1""")
display(waterfall)

In [0]:
final_filter = f"""
FROM {LAB_TBL} l
LEFT JOIN {PRV_TBL} d ON l.NPI_ID = d.NPI_NUMBER
WHERE REGEXP_LIKE(l.PATIENT_ID,'^[0-9]+$') AND l.LCL_LAB_TEST_VAL_TXT IS NOT NULL
  AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%' AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%'
  AND TRIM(l.LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%'
  AND REGEXP_LIKE(TRIM(l.LCL_LAB_TEST_VAL_TXT),'^[0-9]+(\\\\.[0-9]+)?$')
  AND d.PRIMARY_SPECIALTY_CODE IN {SPEC}
  AND ( (UPPER(REPLACE(l.LCL_LAB_TEST_RSLT_UOM_CD_CLEAN,' ','')) IN {UOM}
         AND (UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%GFR%' OR UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%GLOMERULAR%'
              OR UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%ALB%' OR UPPER(l.LCL_LAB_TEST_NM_CLEAN) LIKE '%RATIO%'))
        OR l.LOINC_CD IN ('2823-3','4548-4') )"""

by_year = Get_Data_Snowflakes(f"""
SELECT l.LAB_IDENTIFIER, YEAR({DT}) AS YR, COUNT(*) AS FINAL_ROWS,
       COUNT(DISTINCT l.PATIENT_ID) AS FINAL_PATIENTS, COUNT(DISTINCT l.NPI_ID) AS FINAL_NPIS,
       SUM(IFF(l.LOINC_CD IS NULL,1,0)) AS NULL_LOINC
{final_filter}
GROUP BY 1, 2 ORDER BY 2, 1""")
display(by_year)

patient_overlap = Get_Data_Snowflakes(f"""
SELECT COUNT(DISTINCT IFF(l.LAB_IDENTIFIER='P', l.PATIENT_ID, NULL)) AS P_PATIENTS,
       COUNT(DISTINCT IFF(l.LAB_IDENTIFIER='Q', l.PATIENT_ID, NULL)) AS Q_PATIENTS,
       COUNT(DISTINCT l.PATIENT_ID) AS ALL_PATIENTS
{final_filter}""")
display(patient_overlap)